### RAG Pipeline - Data ingestion to vector DB

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/var/folders/_r/3l8fyfs57f1g8gwg_vpx6svw0000gn/T/ipykernel_43836/3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
def procell_all_pdfs(directory):
    pdf_loaders = [PyPDFLoader]
    all_documents = []
    for pdf_file in Path(directory).glob("*.pdf"):
        for loader_cls in pdf_loaders:
            try:
                loader = loader_cls(str(pdf_file))
                documents = loader.load()
                for doc in documents:
                    doc.metadata["source_file"] = str(pdf_file)
                    doc.metadata['file_type'] = 'pdf'
                all_documents.extend(documents)
            except Exception as e:
                print(f"Failed to load {pdf_file} with {loader_cls}: {e}")
    return all_documents


all_pdf_documents = procell_all_pdfs("../data")
print(all_pdf_documents)

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


# 2. Initialize the splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    length_function=len,
)

# 3. Split the documents
split_docs = splitter.split_documents(all_pdf_documents)

print(f"Total chunks created: {len(split_docs)}")
print(split_docs[0])
print("------")
print(split_docs[-1])


[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '../data/rag.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '../data/rag.pdf', 'file_type': 'pdf'}, page_content='This is a RAG PDF, this will explain the complete process of creating a RAG application\nusing LangChain.\nRAG is a Retrieval-Augmented Generation approach that combines traditional\ninformation retrieval techniques with generative models to provide more accurate and\ncontextually relevant responses.\nThis approach allows the system to leverage both the vast knowledge encoded in the\ngenerative model and the speciﬁc, up-to-date information retrieved from external\nsources, resulting in more reliable and context-aware outputs.\nIn this PDF, we will walk through the steps of setting up a RAG application, including\ndocument loading, text splitting, vector store creation, and query handling using\nLangChain.\nBy the end of this PDF, readers should have a clear under

In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/Users/KumarKrishna/AI_2026/langchain-custom/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1020 05:36:58.530000 43836 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using sentence-transformers"""

    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Model {self.model_name} loaded successfully., dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")


    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if self.model is None:
            self._load_model()
        if self.model is not None:
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"shape of embeddings: {embeddings.shape}")
            return embeddings
        else:
            raise ValueError("Model is not loaded.")

embedding_manager = EmbeddingManager()
embedding_manager

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7751.76it/s]


Model sentence-transformers/all-MiniLM-L6-v2 loaded successfully., dimension: 384


/var/folders/_r/3l8fyfs57f1g8gwg_vpx6svw0000gn/T/ipykernel_43836/558071618.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model {self.model_name} loaded successfully., dimension: {self.model.get_sentence_embedding_dimension()}")


In [5]:
class VectorStore:
    """Manages document embeddings in a chromadb vector store"""

    def __init__(self, collection_name: str, persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )
            self.collection = self.client.get_or_create_collection(self.collection_name)
            print(f"Vector store '{self.collection_name}' initialized successfully at '{self.persist_directory}'")
        except Exception as e:
            print(f"Error initializing vector store '{self.collection_name}': {e}")

    def add_documents(self, documents: List[str], embeddings: np.ndarray):
        if self.collection is None:
            raise ValueError("Vector store is not initialized.")
        try:
            ids =[]
            metadatas = []
            documents_text = []
            embeddings_list = []

            for i, (doc, emb) in enumerate(zip(documents, embeddings)):
                doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
                ids.append(doc_id)

                metadata = dict(doc.metadata)
                metadata['doc_index'] =i
                metadata['content_length'] = len(doc.page_content)
                metadatas.append(metadata)

                documents_text.append(doc.page_content)
                embeddings_list.append(emb.tolist())

            self.collection.add(
                ids=ids,
                documents=documents_text,
                metadatas=metadatas,
                embeddings=embeddings_list
            )
            print(f"Added {len(documents)} documents to the vector store '{self.collection_name}'")
        except Exception as e:
            print(f"Error adding documents to vector store '{self.collection_name}': {e}")

    def query(self, query_embedding: np.ndarray, top_k: int = 5):
        if self.collection is None:
            raise ValueError("Vector store is not initialized.")
        try:
            results = self.collection.query(
                query_embeddings=query_embedding,
                n_results=top_k
            )
            return results
        except Exception as e:
            print(f"Error querying vector store '{self.collection_name}': {e}")
            return None


vector_store = VectorStore(collection_name="rag_pdf")
vector_store


Vector store 'rag_pdf' initialized successfully at '../data/vector_store'


In [6]:
texts= [doc.page_content for doc in split_docs]

embeddings = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(split_docs, embeddings)

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.68it/s]

shape of embeddings: (11, 384)
Added 11 documents to the vector store 'rag_pdf'


In [14]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 1):
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        results = self.vector_store.query(query_embedding, top_k=top_k)
        return results

rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manager=embedding_manager)

response = rag_retriever.retrieve("what is rag ?")
print(response["documents"][0][0])

Batches: 100%|██████████| 1/1 [00:00<00:00, 30.93it/s]

shape of embeddings: (1, 384)
This is a RAG PDF, this will explain the complete process of creating a RAG application


In [15]:
from langchain.agents import create_agent

system_prompt = "You are a helpful assistant that can answer questions." + response["documents"][0][0]


agent = create_agent(
    model="google_genai:gemini-3.8-flash",
    system_prompt=system_prompt,
)

agent.invoke({"messages": [{"role": "user", "content": "What is rag system"}]})

{'messages': [HumanMessage(content='What is rag system', additional_kwargs={}, response_metadata={}, id='27a145bd-3ad1-4110-9ecd-3d23b2ada292'),
  AIMessage(content=[{'type': 'text', 'text': '**RAG** stands for **Retrieval-Augmented Generation**. \n\nIt is an AI architecture that enhances the capabilities of a Large Language Model (LLM) by allowing it to access external, private, or real-time data (such as internal company PDFs, databases, or documentation) before generating a response.\n\n---\n\n### Why do we need RAG?\n\nStandard LLMs (like GPT-4) have significant limitations:\n1. **Knowledge Cutoffs:** They only know what was in their training data up to a specific date.\n2. **Hallucinations:** When they do not know an answer, they might make up convincing but incorrect information.\n3. **No Access to Private Data:** They do not know about your private files, PDFs, or internal company knowledge.\n\n**RAG solves this** by acting like an **"open-book exam"**: instead of forcing the mo